# 06 - Fuentes externas

**Objetivo.** Documentar, cargar y evaluar fuentes externas priorizadas para enriquecer el análisis territorial. Esta etapa separa compatibilidad de integración: primero se verifica cobertura, granularidad y llave de unión.

**Unidad de análisis.** La publicación inmobiliaria activa en cada portal y fecha de consulta (Mercado Libre, ZonaProp, ArgenProp y Airbnb). El barrio es el nivel principal para describir precios, servicios y composición de la oferta; la comuna es el nivel complementario y el común con las fuentes demográficas que no están desagregadas por barrio. De ahí sale la regla de compatibilidad que se aplica a cada fuente:

- Fuente **puntual** (con coordenadas): permite medir distancia por publicación y densidad por barrio, y explica diferencias *dentro* de una misma comuna.
- Fuente **agregada por comuna**: se une por clave `comuna` y solo explica diferencias *entre* comunas.
- Fuente de **geolocalización** (GeoJSON de barrios, USIG): no aporta variables de entorno por sí misma; habilita las uniones anteriores.

Para cada fuente se documenta: fuente concreta, cobertura geográfica, período, granularidad, mecanismo de unión, variable derivada, análisis/KPI/hipótesis al que aporta y limitaciones conocidas. Las hipótesis (H1, H2a, H2b, H3, H4) y los KPIs son los definidos en el informe de la Entrega 1, secciones 4.1 y 4.2.

In [ ]:
from pathlib import Path
import json
import re
import sys
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 80)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Inventario de fuentes externas priorizadas

Las 13 fuentes son las listadas en la sección 3.2 del informe de la Entrega 1. Las de BA Data se cargan desde `Fuentes_de_enriquecimiento/datasets_contextuales_caba.py`. La columna `permite_variacion_intra_comuna` es el criterio de compatibilidad con la unidad de análisis, y `nivel_minimo_compatible` indica hasta qué nivel puede llegar la unión. La prioridad se asigna según cuántas hipótesis dependen de la fuente. `periodo` y `granularidad` se contrastan con los datos reales en la sección de perfilado.

In [ ]:
PERIODO_BADATA = ('Snapshot vigente a la descarga (enlace verificado el 06/09/2026). '
                  'Fecha de ultima actualizacion del recurso: completar desde la metadata del portal.')
UNION_PUNTUAL = ('Dos niveles: (a) por publicacion, distancia haversine desde latitud/longitud del aviso al punto mas cercano '
                 '(BallTree) y conteo en radio; (b) por barrio, asignando cada punto al poligono de F13 y calculando '
                 'densidad = puntos / km2 del barrio.')
LIM_SNAPSHOT = 'Foto vigente a la descarga, frente a avisos de agosto-septiembre de 2026.'

fuentes_externas = pd.DataFrame([
    {
        'id': 'F01', 'fuente': 'Censo Nacional 2022 - Cuadro 2.1 (poblacion, superficie y densidad por comuna)', 'organismo': 'INDEC',
        'referencia': 'Fuentes_de_enriquecimiento/c2022_caba_est_c2_1.xlsx (hoja "Cuadro 2.1"); origen: censo.gob.ar, datos definitivos CABA',
        'cobertura_geografica': 'CABA completa, 15 comunas',
        'periodo': '2022 (datos definitivos del Censo)',
        'granularidad': 'Comuna (agregado)',
        'mecanismo_union': 'Por clave: comuna (entero 1-15, extraido de "Comuna N") contra la columna comuna de la publicacion.',
        'variable_derivada': 'poblacion_total, superficie_km2, densidad_hab_km2; publicaciones_por_1000_hab',
        'aporta_a': ('KPI "Intensidad de oferta publicada en la comuna c" (publicaciones x 1.000 hab.); denominador de H2b; '
                     'contexto demografico para H2a; el stock de hogares/viviendas para H4 requiere otros cuadros del Censo (ver limitaciones).'),
        'limitaciones': ('No distingue barrios dentro de una misma comuna. Solo 15 unidades: poca potencia estadistica y riesgo de falacia ecologica. '
                         'Este cuadro trae poblacion, no hogares ni viviendas: H2a (hogares unipersonales) y los indicadores por 1.000 viviendas de H1/H4 '
                         'necesitan otros cuadros de la misma pagina (a identificar; verificar si alguno llega a nivel barrio). '
                         'Foto de 2022 frente a avisos de 2026.'),
        'nivel_minimo_compatible': 'Comuna', 'permite_variacion_intra_comuna': 'No', 'prioridad': 'Alta',
        'estado': 'Cargada; lista para cruce por comuna',
    },
    {
        'id': 'F02', 'fuente': 'Estaciones de subte', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_subte_estaciones -> subte_estaciones',
        'cobertura_geografica': 'CABA (red de subte; 90 estaciones segun el informe)',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (estacion)',
        'mecanismo_union': UNION_PUNTUAL,
        'variable_derivada': 'dist_subte_m; n_estaciones_500m; densidad_subte_km2_barrio',
        'aporta_a': 'H3 (distancia a subte y precio por m2 / por dormitorio); control de accesibilidad en H1, H2b y H4.',
        'limitaciones': ('Distancia en linea recta, no caminable. No mide frecuencia ni capacidad. Solo cubre CABA. ' + LIM_SNAPSHOT),
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F03', 'fuente': 'Uso (pasajeros) del subte', 'organismo': 'BA Data (GCBA) - organizacion SBASE',
        'referencia': 'https://data.buenosaires.gob.ar/organization/sbase (informe) / link_subte_pasajeros en el .py',
        'cobertura_geografica': 'CABA (red de subte)',
        'periodo': 'Serie desde junio 2013 segun la descripcion del recurso; completar ultimo periodo disponible.',
        'granularidad': ('El informe lo describe por estacion y fecha; el recurso que baja el .py se describe por linea y ano. '
                         'Verificar cual es en el perfilado.'),
        'mecanismo_union': ('Indirecto: estacion mas cercana (F02) -> pasajeros de esa estacion (o de su linea, segun la granularidad real). '
                            'Requiere una llave estacion/linea comun entre ambos datasets.'),
        'variable_derivada': 'pasajeros_estacion_cercana o pasajeros_linea_cercana (usar un periodo reciente y no pandemico)',
        'aporta_a': ('Complemento de H3: intensidad de uso del transporte, no solo cercania. '
                     'La formulacion actual de H3 usa distancia, por lo que esta fuente es opcional.'),
        'limitaciones': ('Si es por linea, no distingue estaciones de la misma linea. 2020-2021 distorsionados por la pandemia. '
                         'El uso del subte no equivale a valoracion residencial. Dificultad de cruce de nombres entre datasets.'),
        'nivel_minimo_compatible': 'Publicacion (via estacion/linea mas cercana)', 'permite_variacion_intra_comuna': 'Parcial', 'prioridad': 'Baja',
        'estado': 'Pendiente de verificar granularidad real',
    },
    {
        'id': 'F04', 'fuente': 'Corredores y estaciones de MetroBus', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_metrobus -> metrobus',
        'cobertura_geografica': 'CABA (corredores de transporte rapido de superficie)',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Corredor/tramo y/o parada (verificar el tipo de geometria en el perfilado)',
        'mecanismo_union': UNION_PUNTUAL + ' Si el recurso es lineal, usar sus vertices o las paradas.',
        'variable_derivada': 'dist_metrobus_m; n_paradas_metrobus_500m',
        'aporta_a': 'H3 (MetroBus como medio diferenciado); H3 compara el efecto por medio de transporte.',
        'limitaciones': ('Distancia a un corredor no equivale a distancia a una parada. No mide frecuencia. '
                         'Cobertura territorial acotada: pocos avisos quedan cerca. ' + LIM_SNAPSHOT),
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F05', 'fuente': 'Estaciones de ferrocarril (Trenes Argentinos)', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_estaciones_ferrocarril -> estaciones_ferrocarril',
        'cobertura_geografica': 'Estaciones ubicadas dentro de CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (estacion)',
        'mecanismo_union': UNION_PUNTUAL,
        'variable_derivada': 'dist_tren_m; n_estaciones_tren_1km',
        'aporta_a': 'H3 (tren como medio diferenciado).',
        'limitaciones': 'Solo estaciones dentro de CABA. No mide frecuencia ni ramal. Pocas estaciones: pocos avisos quedan cerca. ' + LIM_SNAPSHOT,
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F06', 'fuente': 'Hospitales (centros de salud publicos y privados)', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_hospitales -> hospitales',
        'cobertura_geografica': 'CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (institucion)',
        'mecanismo_union': UNION_PUNTUAL,
        'variable_derivada': 'dist_hospital_m; n_hospitales_1km; densidad_hospitales_km2_barrio',
        'aporta_a': 'H2b (servicios sanitarios e intensidad de oferta, normalizada por poblacion); KPI "Densidad de servicios s en el barrio b".',
        'limitaciones': ('La cantidad no mide capacidad ni calidad. Mezcla tipos de establecimiento: conviene distinguirlos si el dataset lo permite. '
                         'El efecto puede ser ambiguo (servicio vs. transito/ruido). ' + LIM_SNAPSHOT),
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F07', 'fuente': 'Establecimientos educativos (publicos y privados)', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_educacion -> educacion',
        'cobertura_geografica': 'CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (establecimiento), con nivel y tipo de gestion',
        'mecanismo_union': UNION_PUNTUAL + ' Filtrar antes por nivel y gestion.',
        'variable_derivada': 'n_educ_500m (por nivel); dist_escuela_m; densidad_educ_km2_barrio',
        'aporta_a': 'H2b (servicios educativos); KPI "Densidad de servicios s en el barrio b".',
        'limitaciones': 'Hay que distinguir niveles y gestion. Mide oferta, no calidad ni matricula. Puede incluir establecimientos cerrados. ' + LIM_SNAPSHOT,
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F08', 'fuente': 'Espacios verdes publicos (plazas, parques y otros)', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_espacios_verdes -> espacios_verdes',
        'cobertura_geografica': 'CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto o poligono segun el recurso (verificar en el perfilado)',
        'mecanismo_union': UNION_PUNTUAL + ' Si hay poligonos, medir contra el borde y no contra el centroide.',
        'variable_derivada': 'dist_verde_m; n_verdes_500m; m2_verde_en_radio (si trae superficie); densidad_verdes_km2_barrio',
        'aporta_a': 'H1 (atractivo turistico: espacios verdes) y H2b (servicios); KPI "Densidad de servicios s en el barrio b".',
        'limitaciones': 'Cercania no mide tamano ni calidad: una plazoleta pesa igual que un parque si solo se cuenta. ' + LIM_SNAPSHOT,
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F09', 'fuente': 'Espacios culturales (publicos y privados)', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_espacios_culturales -> espacios_culturales',
        'cobertura_geografica': 'CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (espacio), con categoria (bares, bibliotecas, centros culturales, cines, teatros, museos, etc.)',
        'mecanismo_union': UNION_PUNTUAL + ' Conteo total y por categoria.',
        'variable_derivada': 'n_cultural_500m; diversidad_categorias_500m; densidad_cultural_km2_barrio',
        'aporta_a': 'H1 (atractivo cultural y turistico, que se espera asociado a mayor densidad de alquiler temporal) y H2b.',
        'limitaciones': ('Mezcla categorias muy distintas (incluye bares). Mide oferta, no flujo de gente. '
                         'Se concentra en el centro y se confunde con centralidad: H1 prevé controlarla. Incluye bibliotecas (ver F10). ' + LIM_SNAPSHOT),
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F10', 'fuente': 'Bibliotecas publicas', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_bibliotecas -> bibliotecas',
        'cobertura_geografica': 'CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (biblioteca)',
        'mecanismo_union': UNION_PUNTUAL,
        'variable_derivada': 'n_bibliotecas_1km; densidad_bibliotecas_km2_barrio',
        'aporta_a': 'Complemento de la dimension cultural de H1 y H2b.',
        'limitaciones': 'Se superpone con F09, que ya incluye bibliotecas: usar una u otra, o deduplicar. Pocos puntos. ' + LIM_SNAPSHOT,
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Baja',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F11', 'fuente': 'Oferta gastronomica turistica (Registro de Prestadores Turisticos)', 'organismo': 'BA Data (GCBA)',
        'referencia': 'link_gastronomia -> gastronomia',
        'cobertura_geografica': 'CABA',
        'periodo': PERIODO_BADATA,
        'granularidad': 'Punto (establecimiento)',
        'mecanismo_union': UNION_PUNTUAL,
        'variable_derivada': 'n_gastro_500m; densidad_gastro_km2_barrio',
        'aporta_a': 'H1 (sitios gastronomicos de interes turistico) y presion potencial del alquiler temporario (H4, como control de atractivo turistico).',
        'limitaciones': ('No es el universo gastronomico: solo los designados de interes turistico / registrados como prestadores. '
                         'Sesgo hacia zonas turisticas y correlacion con centralidad. ' + LIM_SNAPSHOT),
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas); barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'Pendiente de integrar (feature engineering / fusion espacial)',
    },
    {
        'id': 'F12', 'fuente': 'USIG - Normalizador de direcciones (API)', 'organismo': 'USIG (GCBA)',
        'referencia': 'https://servicios.usig.buenosaires.gob.ar/normalizar/',
        'cobertura_geografica': 'CABA',
        'periodo': 'Servicio en linea: se consulta al momento de ejecutar el proceso (registrar la fecha de la corrida).',
        'granularidad': 'Direccion (calle y altura) a punto',
        'mecanismo_union': ('Se envia calle y altura de la publicacion; devuelve la direccion normalizada y sus coordenadas. '
                            'El punto resultante entra luego al mismo flujo que la latitud/longitud original (F13 y fuentes puntuales).'),
        'variable_derivada': 'latitud_usig, longitud_usig, flag_geocodificado_usig',
        'aporta_a': ('Habilitador de H1-H4 para ArgenProp y ZonaProp, que segun el informe no traen latitud, longitud ni barrio '
                     'por el modo --listings-only. Hace posible la fusion espacial a nivel publicacion.'),
        'limitaciones': ('Requiere calle y altura; las direcciones ambiguas o sin altura no geocodifican. Airbnb no publica direccion exacta. '
                         'Verificar limites de uso y el sistema de coordenadas que devuelve. '
                         'Impacto acotado por volumen: ArgenProp y ZonaProp suman 162 y 980 registros.'),
        'nivel_minimo_compatible': 'Publicacion (habilitador)', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Media',
        'estado': 'Pendiente de integrar (fusion espacial)',
    },
    {
        'id': 'F13', 'fuente': 'GeoJSON de barrios', 'organismo': 'GCBA',
        'referencia': 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/innovacion-transformacion-digital/barrios/barrios.geojson',
        'cobertura_geografica': 'CABA, 48 barrios',
        'periodo': 'Division vigente a la descarga; completar fecha de actualizacion desde el portal.',
        'granularidad': 'Poligono (barrio)',
        'mecanismo_union': 'Punto en poligono, con latitud/longitud de la publicacion o del punto externo; la clave resultante es barrio (y comuna si el poligono la trae).',
        'variable_derivada': 'barrio_geo, comuna_geo, area_km2',
        'aporta_a': ('Base de los KPIs por barrio (participacion de la operacion, densidad de servicios, precio por m2, brecha norte-sur) '
                     'y control de superficie en H1 y H2b. Ya se usa para asignar barrio a Airbnb (99,2 % de cobertura segun el informe).'),
        'limitaciones': ('Un punto fuera de CABA o sobre el limite queda sin barrio. Limites en foto vigente. '
                         'El barrio informado por cada portal puede no coincidir con el del poligono: conviene unificar usando el poligono.'),
        'nivel_minimo_compatible': 'Publicacion (si tiene coordenadas) a barrio', 'permite_variacion_intra_comuna': 'Si', 'prioridad': 'Alta',
        'estado': 'En uso para Airbnb; falta aplicarlo al resto de los portales',
    },
])

display(fuentes_externas[['id', 'fuente', 'cobertura_geografica', 'granularidad', 'nivel_minimo_compatible',
                          'permite_variacion_intra_comuna', 'prioridad', 'estado']])
fuentes_externas.to_csv(PROCESSED_DIR / 'fuentes_externas_inventario_v2.csv', index=False)

## Perfilado de las fuentes

Antes de unir nada se verifica con los datos reales lo que el inventario afirma: cuántas filas tiene cada fuente, si trae coordenadas, si esas coordenadas caen dentro de CABA y si trae comuna, barrio o alguna fecha. Esto completa `periodo` y `granularidad` con evidencia y detecta a tiempo problemas de proyección o de encoding.

In [ ]:
BBOX_CABA = {'lat': (-34.72, -34.52), 'lon': (-58.54, -58.33)}  # caja aproximada de CABA en grados (EPSG:4326)
EARTH_R_M = 6_371_000

LAT_CANDS = ['lat', 'latitud', 'latitude', 'y']
LON_CANDS = ['lon', 'long', 'lng', 'longitud', 'longitude', 'x']
GEOM_CANDS = ['geometry', 'wkt', 'the_geom', 'geom', 'geojson']
COMUNA_CANDS = ['comuna', 'comunas', 'nro_comuna', 'numero_comuna']
BARRIO_CANDS = ['barrio', 'barrios', 'nombre_barrio']
FECHA_HINTS = ('fecha', 'anio', 'year', 'actualiz', 'periodo')


def norm_col(c):
    c = unicodedata.normalize('NFKD', str(c)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', c.lower()).strip('_')


def detectar_columna(df, candidatos):
    mapa = {norm_col(c): c for c in df.columns}
    for cand in candidatos:
        if cand in mapa:
            return mapa[cand]
    return None


def a_numero(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False), errors='coerce')


def perfilar_fuente(nombre, df):
    fila = {'dataset': nombre, 'filas': len(df), 'columnas': df.shape[1]}
    col_lat = detectar_columna(df, LAT_CANDS)
    col_lon = detectar_columna(df, LON_CANDS)
    col_geom = detectar_columna(df, GEOM_CANDS)
    fila.update({'col_lat': col_lat, 'col_lon': col_lon, 'col_geom': col_geom})
    if col_lat and col_lon:
        lat, lon = a_numero(df[col_lat]), a_numero(df[col_lon])
        validas = lat.notna() & lon.notna()
        en_caba = lat.between(*BBOX_CABA['lat']) & lon.between(*BBOX_CABA['lon'])
        fila['pct_coords_validas'] = round(validas.mean() * 100, 1)
        fila['pct_en_bbox_caba'] = round(en_caba.mean() * 100, 1)
    else:
        fila['pct_coords_validas'] = np.nan
        fila['pct_en_bbox_caba'] = np.nan
    fila['col_comuna'] = detectar_columna(df, COMUNA_CANDS)
    fila['col_barrio'] = detectar_columna(df, BARRIO_CANDS)
    fila['cols_fecha'] = [c for c in df.columns if any(h in norm_col(c) for h in FECHA_HINTS)]
    fila['columnas_lista'] = list(df.columns)[:25]
    return fila


# El modulo descarga los datasets al importarse (requiere conexion a internet).
sys.path.insert(0, str(EXTERNAL_DIR))
import datasets_contextuales_caba as ctx

datasets_ext = {k: v for k, v in ctx.datasets.items() if k != 'censo_2022'}
no_cargados = [k for k, v in datasets_ext.items() if v is None]
if no_cargados:
    print('NO CARGADOS (revisar link/encoding):', no_cargados)

perfil_fuentes = pd.DataFrame([perfilar_fuente(k, v) for k, v in datasets_ext.items() if v is not None])
display(perfil_fuentes.drop(columns=['columnas_lista']))
perfil_fuentes.to_csv(REPORTS_DIR / 'perfil_fuentes_externas.csv', index=False)

**Lectura del perfilado.** Si `pct_en_bbox_caba` es bajo aunque haya coordenadas, el dataset probablemente viene en una proyección distinta de grados (por ejemplo, metros) y hay que reproyectarlo antes de calcular distancias. Si una fuente no trae coordenadas pero sí comuna o barrio, solo se puede cruzar por esa llave y pasa a ser una fuente agregada. Los hallazgos de esta tabla se vuelcan a las columnas `periodo` y `granularidad` del inventario.

## Carga de Censo 2022 por comuna

In [ ]:
ruta_censo = EXTERNAL_DIR / 'c2022_caba_est_c2_1.xlsx'
if not ruta_censo.exists():
    raise FileNotFoundError(f'No se encontro {ruta_censo}')

censo_2022 = pd.read_excel(
    ruta_censo,
    sheet_name='Cuadro 2.1',
    skiprows=5,
    header=None,
    nrows=16,
    usecols='A:E',
    names=['codigo_comuna_indec', 'comuna_texto', 'superficie_km2', 'poblacion_total', 'densidad_hab_km2'],
)
censo_2022_total_ciudad = censo_2022.iloc[[0]].copy()
censo_2022 = censo_2022.iloc[1:].copy()
censo_2022['comuna'] = censo_2022['comuna_texto'].astype(str).str.extract(r'(\d+)').astype(int)
for col in ['superficie_km2', 'poblacion_total', 'densidad_hab_km2']:
    censo_2022[col] = pd.to_numeric(censo_2022[col], errors='coerce')

# Chequeos de integridad: 15 comunas y suma de poblacion igual al total de la Ciudad
assert len(censo_2022) == 15, 'Se esperaban 15 comunas'
total_ciudad = pd.to_numeric(censo_2022_total_ciudad['poblacion_total'], errors='coerce').iloc[0]
print('Poblacion suma comunas:', int(censo_2022['poblacion_total'].sum()), '| total Ciudad:', total_ciudad)

display(censo_2022)
censo_2022.to_csv(PROCESSED_DIR / 'censo_2022_comunas_v1.csv', index=False)

## Compatibilidad con la base inmobiliaria

In [ ]:
BASE_CONSOLIDADA_CANDIDATES = [
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_outliers_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_post_ajustes_con_flags_nulos_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_flags_nulos_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_v1.csv',
]

DATASETS_PIPELINE = [
    'df_preprocesado_meli_alq',
    'df_preprocesado_meli_alq_temp',
    'df_preprocesado_meli_vtas',
    'df_preprocesado_airbnb_temp',
    'df_preprocesado_argenprop_vtas',
    'df_preprocesado_argenprop_alq',
    'df_preprocesado_zonaprop_vtas',
    'df_preprocesado_zonaprop_alq',
]

CANONICAL_COLS = [
    'id_publicacion', 'fuente_norm', 'fuente', 'tipo_operacion_norm', 'tipo_operacion', 'url', 'fecha_extraccion',
    'scraped_at_utc', '_archivo_origen', 'titulo', 'descripcion', 'tipo_propiedad', 'moneda_norm', 'moneda',
    'precio', 'precio_texto', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado', 'barrio',
    'barrio_norm', 'comuna', 'provincia', 'pais', 'direccion', 'direccion_completa', 'calle', 'altura',
    'latitud', 'longitud', 'superficie_ref_m2', 'ambientes_ref', 'dormitorios_ref', 'banios_ref', 'cocheras',
    'expensas', 'expensas_moneda'
]


def construir_base_consolidada_desde_individuales(processed_dir):
    frames = []
    fuentes_usadas = []

    for name in DATASETS_PIPELINE:
        candidates = [
            processed_dir / f'{name}_con_outliers_v1.csv',
            processed_dir / f'{name}_post_ajustes_con_flags_nulos_v1.csv',
            processed_dir / f'{name}_con_flags_nulos_v1.csv',
            processed_dir / f'{name}_v1.csv',
        ]
        source = next((path for path in candidates if path.exists()), None)
        if source is None:
            continue

        df = pd.read_csv(source, low_memory=False)
        cols = [
            col for col in df.columns
            if col in CANONICAL_COLS or col.endswith('_is_null') or col.endswith('_flag')
        ]
        tmp = df[cols].copy()
        tmp['dataset_preprocesado'] = name
        tmp['_archivo_base_usado'] = source.name
        frames.append(tmp)
        fuentes_usadas.append({'dataset': name, 'archivo': source.name, 'filas': len(tmp)})

    if not frames:
        raise FileNotFoundError('No existe base consolidada ni archivos individuales procesados. Ejecutar notebooks 02-04.')

    return pd.concat(frames, ignore_index=True, sort=False), pd.DataFrame(fuentes_usadas)


base_path = next((path for path in BASE_CONSOLIDADA_CANDIDATES if path.exists()), None)

if base_path is not None:
    df_base = pd.read_csv(base_path, low_memory=False)
    print('Base usada:', base_path.relative_to(REPO_ROOT), df_base.shape)
else:
    df_base, fuentes_base = construir_base_consolidada_desde_individuales(PROCESSED_DIR)
    print('Base construida desde archivos individuales procesados.', df_base.shape)
    display(fuentes_base)

compatibilidad_comuna = pd.DataFrame({
    'metrica': ['filas_totales', 'filas_con_comuna', 'pct_con_comuna', 'comunas_en_base', 'comunas_en_censo'],
    'valor': [
        len(df_base),
        int(df_base['comuna'].notna().sum()) if 'comuna' in df_base.columns else 0,
        round(df_base['comuna'].notna().mean() * 100, 2) if 'comuna' in df_base.columns else 0,
        sorted(df_base['comuna'].dropna().astype(int).unique().tolist()) if 'comuna' in df_base.columns else [],
        sorted(censo_2022['comuna'].dropna().astype(int).unique().tolist()),
    ]
})
display(compatibilidad_comuna)
compatibilidad_comuna.to_csv(REPORTS_DIR / 'compatibilidad_fuentes_externas_comuna.csv', index=False)

### Calidad de las llaves de unión en la base

Las fuentes puntuales dependen de que la publicación tenga coordenadas válidas dentro de CABA; el Censo depende de `comuna`. Según el informe, ArgenProp y ZonaProp casi no traen latitud, longitud ni barrio (se extrajeron con `--listings-only`), por lo que la cobertura se mide **por portal** además del total.

In [ ]:
def pct_no_nulo(col):
    return round(df_base[col].notna().mean() * 100, 2) if col in df_base.columns else np.nan


lat_base = a_numero(df_base['latitud']) if 'latitud' in df_base.columns else pd.Series(np.nan, index=df_base.index)
lon_base = a_numero(df_base['longitud']) if 'longitud' in df_base.columns else pd.Series(np.nan, index=df_base.index)
coords_en_caba = lat_base.between(*BBOX_CABA['lat']) & lon_base.between(*BBOX_CABA['lon'])

llaves_base = pd.DataFrame({
    'llave': ['latitud presente', 'longitud presente', 'coordenadas dentro de CABA', 'barrio_norm presente', 'comuna presente'],
    'pct_filas': [pct_no_nulo('latitud'), pct_no_nulo('longitud'), round(coords_en_caba.mean() * 100, 2),
                  pct_no_nulo('barrio_norm'), pct_no_nulo('comuna')],
    'habilita': ['union espacial', 'union espacial', 'union espacial (sin coordenadas fuera de CABA)',
                 'union por barrio', 'union por comuna (Censo)'],
})
display(llaves_base)
llaves_base.to_csv(REPORTS_DIR / 'llaves_union_base_inmobiliaria.csv', index=False)

# Cobertura por portal y operacion: define que fuentes puntuales pueden unirse a cada grupo
col_portal = 'fuente_norm' if 'fuente_norm' in df_base.columns else ('fuente' if 'fuente' in df_base.columns else None)
col_oper = 'tipo_operacion_norm' if 'tipo_operacion_norm' in df_base.columns else ('tipo_operacion' if 'tipo_operacion' in df_base.columns else None)
if col_portal:
    claves = [c for c in [col_portal, col_oper] if c]
    tmp_llaves = df_base[claves].copy()
    tmp_llaves['con_coords_caba'] = coords_en_caba.values
    tmp_llaves['con_barrio'] = df_base['barrio_norm'].notna().values if 'barrio_norm' in df_base.columns else False
    tmp_llaves['con_comuna'] = df_base['comuna'].notna().values if 'comuna' in df_base.columns else False
    llaves_por_portal = (tmp_llaves.groupby(claves)
                         .agg(filas=('con_comuna', 'size'),
                              pct_coords_caba=('con_coords_caba', lambda s: round(s.mean() * 100, 1)),
                              pct_barrio=('con_barrio', lambda s: round(s.mean() * 100, 1)),
                              pct_comuna=('con_comuna', lambda s: round(s.mean() * 100, 1)))
                         .reset_index())
    display(llaves_por_portal)
    llaves_por_portal.to_csv(REPORTS_DIR / 'llaves_union_por_portal.csv', index=False)

## Unión espacial y variación intra-comuna

Funciones para derivar las variables de las fuentes puntuales. Después se mide, para cada fuente, qué porcentaje de la varianza de la distancia ocurre *dentro* de las comunas: si es alto, la fuente aporta información que el Censo (agregado por comuna, 0 % por construcción) no puede dar. Es la evidencia de compatibilidad con la unidad de análisis.

In [ ]:
from sklearn.neighbors import BallTree


def _arbol(df_ext, lat_ext, lon_ext):
    ext = pd.DataFrame({'lat': a_numero(df_ext[lat_ext]), 'lon': a_numero(df_ext[lon_ext])}).dropna()
    ext = ext[ext['lat'].between(*BBOX_CABA['lat']) & ext['lon'].between(*BBOX_CABA['lon'])]
    return BallTree(np.radians(ext[['lat', 'lon']].values), metric='haversine')


def _coords_aviso(df_prop, lat_prop, lon_prop):
    p = pd.DataFrame({'lat': a_numero(df_prop[lat_prop]), 'lon': a_numero(df_prop[lon_prop])}).dropna()
    return p[p['lat'].between(*BBOX_CABA['lat']) & p['lon'].between(*BBOX_CABA['lon'])]


def distancia_minima_m(df_prop, df_ext, lat_ext, lon_ext, lat_prop='latitud', lon_prop='longitud'):
    tree = _arbol(df_ext, lat_ext, lon_ext)
    p = _coords_aviso(df_prop, lat_prop, lon_prop)
    out = pd.Series(np.nan, index=df_prop.index)
    dist, _ = tree.query(np.radians(p.values), k=1)
    out.loc[p.index] = dist[:, 0] * EARTH_R_M
    return out


def conteo_en_radio(df_prop, df_ext, lat_ext, lon_ext, radio_m, lat_prop='latitud', lon_prop='longitud'):
    tree = _arbol(df_ext, lat_ext, lon_ext)
    p = _coords_aviso(df_prop, lat_prop, lon_prop)
    out = pd.Series(np.nan, index=df_prop.index)
    out.loc[p.index] = tree.query_radius(np.radians(p.values), r=radio_m / EARTH_R_M, count_only=True)
    return out


def pct_varianza_intra(df, valor, grupo):
    d = df[[valor, grupo]].dropna()
    if len(d) < 2:
        return np.nan
    media = d[valor].mean()
    sst = ((d[valor] - media) ** 2).sum()
    ssb = d.groupby(grupo)[valor].apply(lambda s: len(s) * (s.mean() - media) ** 2).sum()
    return round(100 * (1 - ssb / sst), 2) if sst > 0 else np.nan


resultados = []
for nombre, df_ext in datasets_ext.items():
    if df_ext is None:
        continue
    lat_ext = detectar_columna(df_ext, LAT_CANDS)
    lon_ext = detectar_columna(df_ext, LON_CANDS)
    if not (lat_ext and lon_ext):
        resultados.append({'fuente': nombre, 'estado': 'sin coordenadas: solo cruce por comuna/barrio'})
        continue
    try:
        dist = distancia_minima_m(df_base, df_ext, lat_ext, lon_ext)
    except Exception as exc:  # por ejemplo, coordenadas fuera de CABA o en otra proyeccion
        resultados.append({'fuente': nombre, 'estado': f'error: {exc}'})
        continue
    tmp = pd.DataFrame({'dist': dist, 'comuna': df_base['comuna'] if 'comuna' in df_base.columns else np.nan})
    resultados.append({
        'fuente': nombre,
        'estado': 'ok',
        'pct_avisos_con_distancia': round(dist.notna().mean() * 100, 1),
        'dist_mediana_m': round(dist.median(), 0),
        'dist_p90_m': round(dist.quantile(0.9), 0),
        'pct_varianza_intra_comuna': pct_varianza_intra(tmp, 'dist', 'comuna'),
    })

compat_espacial = pd.DataFrame(resultados)
compat_espacial = pd.concat([
    compat_espacial,
    pd.DataFrame([{'fuente': 'censo_2022', 'estado': 'ok (union por comuna)', 'pct_varianza_intra_comuna': 0.0}]),
], ignore_index=True)
display(compat_espacial)
compat_espacial.to_csv(REPORTS_DIR / 'compatibilidad_fuentes_externas_espacial.csv', index=False)

## Unión por barrio: densidad de servicios

El informe define el KPI *Densidad de servicios s en el barrio b = puntos del servicio s / superficie del barrio b (km²)*. Se calcula asignando cada punto externo al polígono de barrio (F13) con una función propia de punto en polígono (no requiere GeoPandas), y la superficie se calcula con una proyección local equirrectangular, suficiente a la escala de CABA. Después se mide qué parte de la variación de la densidad ocurre entre barrios de una misma comuna.

In [ ]:
import urllib.request


def _anillos(geom):
    """Anillos exteriores de un Polygon o MultiPolygon (los huecos se ignoran)."""
    if geom['type'] == 'Polygon':
        return [geom['coordinates'][0]]
    if geom['type'] == 'MultiPolygon':
        return [poly[0] for poly in geom['coordinates']]
    return []


def _en_anillo(lon, lat, anillo):
    xs = np.array([c[0] for c in anillo])
    ys = np.array([c[1] for c in anillo])
    dentro = np.zeros(len(lon), dtype=bool)
    j = len(xs) - 1
    for i in range(len(xs)):
        cruza = ((ys[i] > lat) != (ys[j] > lat)) & (lon < (xs[j] - xs[i]) * (lat - ys[i]) / (ys[j] - ys[i] + 1e-300) + xs[i])
        dentro ^= cruza
        j = i
    return dentro


def _area_km2(anillo):
    lat0 = np.radians(np.mean([c[1] for c in anillo]))
    x = np.array([np.radians(c[0]) * EARTH_R_M * np.cos(lat0) for c in anillo])
    y = np.array([np.radians(c[1]) * EARTH_R_M for c in anillo])
    return abs(0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)) / 1e6


def tabla_barrios(geojson):
    barrios = []
    for feat in geojson['features']:
        props = {norm_col(k): v for k, v in feat['properties'].items()}
        anillos = _anillos(feat['geometry'])
        barrios.append({
            'barrio': next((props[k] for k in ('barrio', 'nombre', 'name') if k in props), None),
            'comuna': next((props[k] for k in ('comuna', 'comunas') if k in props), np.nan),
            'area_km2': sum(_area_km2(a) for a in anillos),
            '_anillos': anillos,
        })
    return pd.DataFrame(barrios)


def asignar_barrio(lat, lon, barrios):
    """Devuelve el nombre de barrio de cada punto (None si cae fuera de todos los poligonos)."""
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    asignado = np.full(len(lat), -1)
    validos = np.isfinite(lat) & np.isfinite(lon)
    for k, anillos in enumerate(barrios['_anillos']):
        for anillo in anillos:
            pendientes = validos & (asignado == -1)
            if pendientes.any():
                asignado[pendientes] = np.where(_en_anillo(lon[pendientes], lat[pendientes], anillo), k, -1)
    return [barrios['barrio'].iloc[k] if k >= 0 else None for k in asignado]

In [ ]:
URL_BARRIOS = 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/innovacion-transformacion-digital/barrios/barrios.geojson'

try:
    with urllib.request.urlopen(URL_BARRIOS, timeout=60) as resp:
        geo_barrios = json.load(resp)
    barrios = tabla_barrios(geo_barrios)
    print(f'Barrios cargados: {len(barrios)} (se esperaban 48). Area total: {barrios["area_km2"].sum():.1f} km2')
except Exception as exc:
    barrios = None
    print('No se pudo cargar el GeoJSON de barrios:', exc)

if barrios is not None:
    conteos = {}
    for nombre, df_ext in datasets_ext.items():
        if df_ext is None:
            continue
        lat_ext = detectar_columna(df_ext, LAT_CANDS)
        lon_ext = detectar_columna(df_ext, LON_CANDS)
        if not (lat_ext and lon_ext):
            continue
        asignado = pd.Series(asignar_barrio(a_numero(df_ext[lat_ext]), a_numero(df_ext[lon_ext]), barrios))
        conteos[nombre] = asignado.value_counts()
        print(f'{nombre}: {asignado.notna().mean() * 100:.1f} % de los puntos asignados a un barrio')

    densidad = pd.DataFrame(conteos).reindex(barrios['barrio']).fillna(0)
    densidad = densidad.div(barrios.set_index('barrio')['area_km2'], axis=0)
    densidad.columns = [f'densidad_{c}_km2' for c in densidad.columns]
    densidad['comuna'] = barrios.set_index('barrio')['comuna']
    densidad['area_km2'] = barrios.set_index('barrio')['area_km2']
    display(densidad.round(2))
    densidad.to_csv(PROCESSED_DIR / 'densidad_servicios_por_barrio_v1.csv')

    intra_barrio = pd.DataFrame([
        {'servicio': c, 'pct_varianza_entre_barrios_de_misma_comuna': pct_varianza_intra(densidad.reset_index(), c, 'comuna')}
        for c in densidad.columns if c.startswith('densidad_')
    ])
    display(intra_barrio)
    intra_barrio.to_csv(REPORTS_DIR / 'compatibilidad_fuentes_externas_barrio.csv', index=False)

## Cierre de fuentes externas

Para esta entrega no es obligatorio integrar todas las fuentes. Queda documentado qué aporta cada una, con qué granularidad y mecanismo de unión, y qué limitaciones tiene.

| Fuente(s) | Nivel de unión | Explica diferencias dentro de una comuna | Hipótesis / KPI |
|---|---|---|---|
| Censo 2022, Cuadro 2.1 (F01) | Comuna | No | Intensidad de oferta por 1.000 hab.; H2b; contexto de H2a |
| Subte, MetroBus, tren (F02, F04, F05) | Publicación y barrio | Sí | H3 y control de accesibilidad en H1, H2b, H4 |
| Educación, hospitales (F06, F07) | Publicación y barrio | Sí | H2b |
| Verdes, culturales, bibliotecas, gastronomía (F08 a F11) | Publicación y barrio | Sí | H1, H2b y control de atractivo en H4 |
| Uso del subte (F03) | Estación o línea | Parcial | Complemento de H3 |
| GeoJSON de barrios y USIG (F13, F12) | Habilitadores | Sí | KPIs por barrio; coordenadas para ArgenProp y ZonaProp |

**Pendientes para completar:**
- H2a necesita hogares unipersonales y H1/H4 necesitan stock de viviendas: el Cuadro 2.1 no los trae, hay que identificar otros cuadros del Censo en la página de datos definitivos de CABA (y si alguno llega a barrio).
- ArgenProp y ZonaProp dependen de F12 (USIG) para tener coordenadas; hasta entonces solo pueden unirse por comuna o por el barrio que ya traigan.
- Completar `periodo` (fecha de actualización de cada recurso) y `granularidad` (puntos o polígonos) con el perfilado.

**Limitaciones transversales:**
1. Las fuentes son un snapshot actual y los avisos son de agosto-septiembre de 2026: puede haber un desfasaje temporal pequeño.
2. Las distancias son en línea recta, no caminables.
3. Varias variables (subte, culturales, gastronomía) correlacionan con la centralidad. El informe prevé controlarla en H1 a H4; al modelar hay riesgo de multicolinealidad.
4. Contar puntos no mide calidad, capacidad ni uso.
5. Corte transversal y observacional: las asociaciones no son causales.